# **Day 71: Multiprocessing (Doing Many Things Together, Part 2)**

Yesterday we saw that threads do not make **heavy calculations** faster.

**Multiprocessing** starts **separate Python programs** (called **processes**). Each process runs on a **different CPU core**, so heavy calculations really happen at the same time.

**Real-life example:**
- **Threads** = one cook doing many jobs by switching between them while waiting.
- **Processes** = many cooks, each in their own kitchen, all cooking at the same time.

In [1]:
import os
print("CPU cores on this computer:", os.cpu_count())

CPU cores on this computer: 20


## **Important Note for Jupyter on Windows**
On Windows, each new process starts a **fresh Python** and **imports** the function it needs. A function written directly in a notebook cell cannot be imported this way.

**The fix:** Write the worker functions in a separate `.py` file and import them. (In a normal `.py` script, we also need the `if __name__ == "__main__":` line.)

In [2]:
%%writefile workers.py
import math

def count_primes(limit):
    """Count the prime numbers below limit. This is slow on purpose (heavy work)."""
    count = 0
    for n in range(2, limit):
        is_prime = True
        for d in range(2, int(math.isqrt(n)) + 1):
            if n % d == 0:
                is_prime = False
                break
        count += is_prime
    return count

def tile_ndvi_stats(args):
    """Make a fake tile of pixels, calculate NDVI and return some statistics."""
    tile_id, size, seed = args
    import random
    rnd = random.Random(seed)
    values = []
    for _ in range(size):
        red = rnd.uniform(0.02, 0.20)
        nir = rnd.uniform(0.05, 0.50)
        values.append((nir - red) / (nir + red))
    vegetation = sum(1 for v in values if v > 0.5)
    return tile_id, round(sum(values) / size, 3), round(vegetation / size * 100, 1)

Writing workers.py


## **Comparing: One by One vs Many Processes**

In [3]:
import time
from concurrent.futures import ProcessPoolExecutor
from workers import count_primes

jobs = [600_000, 650_000, 700_000, 750_000]

start = time.perf_counter()
one_by_one = [count_primes(n) for n in jobs]
time_one = time.perf_counter() - start
print("One by one :", one_by_one, f"{time_one:.2f} seconds")

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    parallel = list(executor.map(count_primes, jobs))
time_parallel = time.perf_counter() - start
print("4 processes:", parallel, f"{time_parallel:.2f} seconds")

print(f"Speed-up: {time_one / time_parallel:.1f} times faster")

One by one : [49098, 52831, 56543, 60238] 4.06 seconds


4 processes: [49098, 52831, 56543, 60238] 1.37 seconds
Speed-up: 3.0 times faster


The speed-up is not exactly 4 times, because starting new processes takes some time, and the four jobs are not the same size.

## **Example from Remote Sensing: Processing Tiles in Parallel**
A big satellite image can be cut into **tiles** (Day 54). Each tile can be processed **separately**, so this is a perfect job for multiprocessing: give each CPU core a different tile.

In [4]:
from workers import tile_ndvi_stats

tile_jobs = [(f"tile_{i:02d}", 200_000, i) for i in range(8)]

start = time.perf_counter()
with ProcessPoolExecutor() as executor:
    results = list(executor.map(tile_ndvi_stats, tile_jobs))
print(f"8 tiles processed in {time.perf_counter() - start:.2f} seconds\n")

print(f"{'Tile':<9}{'Mean NDVI':>10}{'Vegetation %':>14}")
for tile_id, mean, veg in results:
    print(f"{tile_id:<9}{mean:>10}{veg:>14}")

8 tiles processed in 0.28 seconds

Tile      Mean NDVI  Vegetation %
tile_00       0.381          40.0
tile_01        0.38          39.9
tile_02        0.38          39.8
tile_03        0.38          40.0
tile_04       0.379          39.6
tile_05       0.378          39.6
tile_06        0.38          39.7
tile_07        0.38          39.8


(The pixel values are random, only to show the idea. On Day 81 we will see that the **NumPy** library can do this kind of calculation much faster, even on one core.)

## **multiprocessing.Pool: The Older Way**

In [5]:
import multiprocessing as mp
from workers import count_primes

with mp.Pool(4) as pool:
    print(pool.map(count_primes, [100, 1000, 10000]))

[25, 168, 1229]


## **What Can Be Sent to a Process?**
Everything sent to a process is **copied** by a method called **pickling**. Lambda functions and functions made inside a notebook **cannot** be pickled.

In [6]:
import pickle
try:
    pickle.dumps(lambda x: x)
except Exception as e:
    print(type(e).__name__, "-", e)

PicklingError - Can't pickle <function <lambda> at 0x000001BFBDE18510>: it's not found as __main__.<lambda>


## **Which One Should I Use?**
| | Threads | Processes | asyncio (next day) |
|---|---|---|---|
| Best for | Waiting (downloads, files) | Heavy calculations | Very many waiting jobs (1000s of web requests) |
| Memory | Shared | Separate (copied) | Shared |
| Start-up cost | Low | High | Very low |
| Example | Download 20 tiles | Process 20 tiles | Check 1000 web pages |

In [7]:
# Quick question
# You need to calculate NDVI for 50 large images. Threads or processes?